In [8]:
import pandas as pd
from pathlib import Path   
import numpy as np

In [9]:
result_path = Path("../results/decomposition")
# bias_types = ["less_negative_class", "less_positive_class", "mean_difference"]
bias_types = ["less_positive_class"]
less_bias_strengths = ["0.1"]
mean_bias_strengthts = ["0.8"]
method_name_replacer = {"uniform": "Uniform", "kmm": "KMM", "psa": "PSA", "mrs-forest": "MRS",
                            # "soft-mrs-exponential": "Soft-MRS-Exponential",
                               "mrs-forest": "MRS", 
                       # "fw-mrs-temperature-svm": "FW-MRS-SVM", 
                        "fw-mrs-temperature": "FW-MRS-RF",
                        "fw-mrs-temperature-negative": "FW-MRS$_{Neg}$",
                          }
data_set_replacer = {"folktables_employment": "Employment", "folktables_income": "Income",
                               "breast_cancer": "Breast Cancer", "hr_analytics": "HR Analytic", "loan_prediction": "Loan",
                               "diabetes": "Diabetes", "german_credit": "German Credit", "bank_marketing": "Bank Marketing"}

In [10]:
aurocs = []
auprcs = []
dict_list = []
for dataset in data_set_replacer.keys():
    for bias_type in bias_types:
        for method in method_name_replacer.keys():
            if bias_type == "mean_difference":
                bias_strengths = mean_bias_strengthts
            else : 
                bias_strengths = less_bias_strengths
            for bias_strength in bias_strengths:
                json_file = result_path / dataset / bias_type /  bias_strength/ method / "bias_variance_decomposition.json"
                try:
                    result_file = pd.read_json(str(json_file))
                except FileNotFoundError:
                    continue
                dict_list.append(
                    {
                        "Method": method, "Data Set": dataset, 
                        "0-1 Bias": result_file["0-1 loss"]["average_bias"], 
                        "0-1 Variance": result_file["0-1 loss"]["average variance"], 
                        "0-1 Expexted Loss": result_file["0-1 loss"]["average expected loss"],
                        "MSE Bias": result_file["mse loss"]["average_bias"], 
                        "MSE Variance": result_file["mse loss"]["average variance"], 
                        "MSE Expexted Loss": result_file["mse loss"]["average expected loss"],
                        "Bias Type": bias_type, "Bias Strength": bias_strength,
                    }
                                )
result_df = pd.DataFrame(data=dict_list)

In [11]:
result_df = result_df.replace(method_name_replacer)
result_df = result_df.replace(data_set_replacer)
result_df


,Method,Data Set,0-1 Bias,0-1 Variance,0-1 Expexted Loss,MSE Bias,MSE Variance,MSE Expexted Loss,Bias Type,Bias Strength
0,Uniform,Employment,0.211500,0.157220,0.264080,0.164226,0.014898,0.179124,less_positive_class,0.1
1,KMM,Employment,0.261500,0.200220,0.302880,0.173607,0.021788,0.195394,less_positive_class,0.1
2,PSA,Employment,0.212500,0.143360,0.250840,0.158117,0.012214,0.170331,less_positive_class,0.1
3,MRS,Employment,0.216000,0.148980,0.257700,0.160596,0.011898,0.172493,less_positive_class,0.1
4,FW-MRS-RF,Employment,0.215500,0.141920,0.252680,0.159823,0.008155,0.167978,less_positive_class,0.1
5,FW-MRS$_{Neg}$,Employment,0.224000,0.158420,0.267280,0.161246,0.017675,0.178921,less_positive_class,0.1
6,Uniform,Income,0.292000,0.133660,0.307180,0.188564,0.017233,0.205797,less_positive_class,0.1
7,KMM,Income,0.252000,0.256330,0.329710,0.187343,0.036644,0.223986,less_positive_class,0.1
8,PSA,Income,0.288000,0.217420,0.329120,0.196681,0.032110,0.228791,less_positive_class,0.1
9,MRS,Income,0.278500,0.143800,0.301200,0.185063,0.017663,0.202726,less_positive_class,0.1


In [12]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            bias_01_values = []
            for dataset in data_set_replacer.values():
                bias_01 = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["0-1 Bias"].iloc[0]
                bias_01_values.append(round(bias_01, 3))


            print(f"\t& {method} & {bias_01_values[0]} & {bias_01_values[1]} & {bias_01_values[2]} & {bias_01_values[3]} & {bias_01_values[4]} \
& {bias_01_values[5]} & {bias_01_values[6]} & {bias_01_values[7]} & \\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & 0.212 & 0.292 & 0.048 & 0.165 & 0.694 & 0.135 & 0.299 & 0.116 & \\
	& KMM & 0.262 & 0.252 & 0.031 & 0.165 & 0.694 & 0.135 & 0.299 & 0.116 & \\
	& PSA & 0.213 & 0.288 & 0.035 & 0.165 & 0.694 & 0.135 & 0.299 & 0.116 & \\
	& MRS & 0.216 & 0.278 & 0.035 & 0.165 & 0.694 & 0.135 & 0.299 & 0.116 & \\
	& FW-MRS-RF & 0.216 & 0.284 & 0.031 & 0.165 & 0.694 & 0.135 & 0.299 & 0.116 & \\
	& FW-MRS$_{Neg}$ & 0.224 & 0.296 & 0.039 & 0.162 & 0.694 & 0.135 & 0.299 & 0.116 & \\




In [13]:
for bias_type in bias_types:
    if bias_type == "mean_difference":
        bias_strengths = mean_bias_strengthts
    else : 
        bias_strengths = less_bias_strengths
    for bias_strength in bias_strengths:
        print(f"{bias_type}, {bias_strength}")
        for method in result_df["Method"].unique():
            variance_01_values = []
            for dataset in data_set_replacer.values():
                variance_01 = result_df.loc[(result_df["Method"]==method) & (result_df["Bias Type"]==bias_type) & 
                                                    (result_df["Bias Strength"]==bias_strength) & 
                                                    (result_df["Data Set"]==dataset)]["0-1 Variance"].iloc[0]
                variance_01_values.append(np.round(variance_01, 3))

            print(f"\t& {method} & {variance_01_values[0]} & {variance_01_values[1]} & {variance_01_values[2]} & {variance_01_values[3]} & \
{variance_01_values[4]} & {variance_01_values[5]} & {variance_01_values[6]} & {variance_01_values[7]} &\\\\")
        print("\n")

less_positive_class, 0.1
	& Uniform & 0.157 & 0.134 & 0.104 & 0.072 & 0.066 & 0.076 & 0.014 & 0.014 &\\
	& KMM & 0.2 & 0.256 & 0.077 & 0.157 & 0.091 & 0.053 & 0.02 & 0.034 &\\
	& PSA & 0.143 & 0.217 & 0.04 & 0.109 & 0.064 & 0.094 & 0.025 & 0.027 &\\
	& MRS & 0.149 & 0.144 & 0.02 & 0.086 & 0.085 & 0.074 & 0.01 & 0.012 &\\
	& FW-MRS-RF & 0.142 & 0.134 & 0.036 & 0.053 & 0.072 & 0.101 & 0.01 & 0.01 &\\
	& FW-MRS$_{Neg}$ & 0.158 & 0.139 & 0.028 & 0.088 & 0.099 & 0.08 & 0.021 & 0.01 &\\




In [14]:
result_df["Rank Bias"] = result_df.round(3).groupby("Data Set")["0-1 Bias"].rank(ascending=True)
result_df["Rank Variance"] = result_df.round(3).groupby("Data Set")["0-1 Variance"].rank(ascending=True)
result_df[["Method", "Rank Bias", "Rank Variance"]].groupby("Method").mean()

,Rank Bias,Rank Variance
Method,,
FW-MRS$_{Neg}$,3.8750,3.8125
FW-MRS-RF,3.2500,2.3125
KMM,3.3125,4.8750
MRS,3.3750,2.6875
PSA,3.4375,4.1250
Uniform,3.7500,3.1875
